In [11]:
### Project Paths
from pathlib import Path
import sys
import importlib
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image, ImageChops
import numpy as np

# Change this depending on notebook location:
# 0 = notebook is 1 folder inside project root, e.g. dp_termpaper/edu1
# 1 = notebook is 2 folders inside project root, e.g. dp_termpaper/data/moments
# 2 = notebook is 3 folders inside project root
amount_of_levels = 0

DIR = Path.cwd().resolve().parents[amount_of_levels]

if str(DIR) not in sys.path:
    sys.path.insert(0, str(DIR))

import project_paths as pp
from project_imports import *

importlib.reload(pp)
jax.config.update("jax_enable_x64", True)

In [12]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

new_model_dirs = {
    "Education 1": pp.SIM_PLOTS_DIR / "edu1_phi_specification",
    "Education 2": pp.SIM_PLOTS_DIR / "edu2_phi_specification",
    "Education 3": pp.SIM_PLOTS_DIR / "edu3_phi_specification",
}

old_model_dirs = {
    "Education 1": pp.SIM_PLOTS_DIR / "original_edu1_specification",
    "Education 2": pp.SIM_PLOTS_DIR / "original_edu2_specification",
    "Education 3": pp.SIM_PLOTS_DIR / "original_edu3_specification",
}

output_dir = pp.SIM_PLOTS_DIR / "combined_moment_comparisons"
output_dir.mkdir(parents=True, exist_ok=True)

In [13]:

# # ------------------------------------------------------------
# # Moments to combine
# # ------------------------------------------------------------
# moments = [
#     "avg_experience_over_age",
#     "avg_hours_over_age",
#     "avg_wage_over_age",
#     "avg_wealth_over_age",
#     "hours_0_over_age",
#     "hours_1_over_age",
#     "hours_2_over_age",
#     "hours_3_over_age",
#     "hours_4_over_age",
#     "nowork_nowork_over_age",
#     "work_work_over_age",
#     "prob_work_over_age",
# ]

# # Nice titles for each moment
# moment_titles = {
#     "avg_experience_over_age": "Average Experience",
#     "avg_hours_over_age": "Average Hours",
#     "avg_wage_over_age": "Average Wage",
#     "avg_wealth_over_age": "Average Wealth",
#     "hours_0_over_age": "Share Choosing 0 Hours",
#     "hours_1_over_age": "Share Choosing Hours Category 1",
#     "hours_2_over_age": "Share Choosing Hours Category 2",
#     "hours_3_over_age": "Share Choosing Hours Category 3",
#     "hours_4_over_age": "Share Choosing Hours Category 4",
#     "nowork_nowork_over_age": "No-work to No-work Transition",
#     "work_work_over_age": "Work to Work Transition",
#     "prob_work_over_age": "Probability of Working",
# }

# # ------------------------------------------------------------
# # Helper function
# # ------------------------------------------------------------
# def add_image_to_axis(ax, image_path):
#     if image_path.exists():
#         img = mpimg.imread(image_path)
#         ax.imshow(img)
#         ax.axis("off")
#     else:
#         ax.text(
#             0.5,
#             0.5,
#             f"Missing file:\n{image_path.name}",
#             ha="center",
#             va="center",
#             fontsize=10,
#         )
#         ax.axis("off")


# # ------------------------------------------------------------
# # Create one 2 x 3 figure per moment
# # ------------------------------------------------------------
# for moment in moments:
#     fig, axes = plt.subplots(
#         2,
#         3,
#         figsize=(14, 8),
#     )

#     fig.suptitle(
#         moment_titles.get(moment, moment.replace("_", " ").title()),
#         fontsize=18,
#         y=0.98,
#     )

#     # Top row: new model
#     for col_idx, (edu_label, folder) in enumerate(new_model_dirs.items()):
#         image_path = folder / f"{moment}.png"
#         add_image_to_axis(axes[0, col_idx], image_path)

#         axes[0, col_idx].set_title(
#             edu_label,
#             fontsize=13,
#             pad=8,
#         )

#     # Bottom row: old model
#     for col_idx, (edu_label, folder) in enumerate(old_model_dirs.items()):
#         image_path = folder / f"{moment}.png"
#         add_image_to_axis(axes[1, col_idx], image_path)

#     # Row labels
#     fig.text(
#         0.03,
#         0.69,
#         "Extended model",
#         va="center",
#         ha="center",
#         rotation=90,
#         fontsize=14,
#         fontweight="bold",
#     )

#     fig.text(
#         0.03,
#         0.29,
#         "Baseline model",
#         va="center",
#         ha="center",
#         rotation=90,
#         fontsize=14,
#         fontweight="bold",
#     )

#     plt.tight_layout(rect=[0.06, 0.02, 1, 0.94])

#     output_path = output_dir / f"{moment}_comparison_2x3.png"
#     fig.savefig(output_path, dpi=300, bbox_inches="tight")
#     plt.close(fig)

#     print(f"Saved: {output_path}")

In [14]:
moments = [
    "avg_experience_over_age",
    "avg_hours_over_age",
    "avg_wage_over_age",
    "avg_wealth_over_age",
    "hours_0_over_age",
    "hours_1_over_age",
    "hours_2_over_age",
    "hours_3_over_age",
    "hours_4_over_age",
    "nowork_nowork_over_age",
    "work_work_over_age",
    "prob_work_over_age",
]

moment_titles = {
    "avg_experience_over_age": "Average Experience",
    "avg_hours_over_age": "Average Hours",
    "avg_wage_over_age": "Average Wage",
    "avg_wealth_over_age": "Average Wealth",
    "hours_0_over_age": "Share Choosing 0 Hours",
    "hours_1_over_age": "Share Choosing Hours Category 1",
    "hours_2_over_age": "Share Choosing Hours Category 2",
    "hours_3_over_age": "Share Choosing Hours Category 3",
    "hours_4_over_age": "Share Choosing Hours Category 4",
    "nowork_nowork_over_age": "No-work to No-work Transition",
    "work_work_over_age": "Work to Work Transition",
    "prob_work_over_age": "Probability of Working",
}


def crop_white_margin(image_path, padding=8):
    """
    Crops white margins from a saved matplotlib PNG.
    Keeps a small padding so axis labels are not clipped.
    """
    img = Image.open(image_path).convert("RGB")

    bg = Image.new("RGB", img.size, (255, 255, 255))
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()

    if bbox is None:
        return np.asarray(img)

    left, upper, right, lower = bbox

    left = max(left - padding, 0)
    upper = max(upper - padding, 0)
    right = min(right + padding, img.size[0])
    lower = min(lower + padding, img.size[1])

    cropped = img.crop((left, upper, right, lower))
    return np.asarray(cropped)


def add_png(ax, image_path):
    ax.axis("off")

    if not image_path.exists():
        ax.text(
            0.5,
            0.5,
            f"Missing:\n{image_path.name}",
            ha="center",
            va="center",
            fontsize=9,
        )
        return

    img = crop_white_margin(image_path, padding=6)
    ax.imshow(img)
    ax.set_anchor("C")


for moment in moments:
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(13.5, 7.2),
        constrained_layout=False,
    )

    fig.suptitle(
        moment_titles.get(moment, moment.replace("_", " ").title()),
        fontsize=17,
        fontweight="normal",
        y=0.965,
    )

    for col_idx, (edu_label, folder) in enumerate(new_model_dirs.items()):
        image_path = folder / f"{moment}.png"
        add_png(axes[0, col_idx], image_path)
        axes[0, col_idx].set_title(
            edu_label,
            fontsize=12,
            fontweight="normal",
            pad=4,
        )

    for col_idx, (edu_label, folder) in enumerate(old_model_dirs.items()):
        image_path = folder / f"{moment}.png"
        add_png(axes[1, col_idx], image_path)

    fig.text(
        0.025,
        0.66,
        "Extended model",
        va="center",
        ha="center",
        rotation=90,
        fontsize=12,
        fontweight="normal",
    )

    fig.text(
        0.025,
        0.285,
        "Baseline model",
        va="center",
        ha="center",
        rotation=90,
        fontsize=12,
        fontweight="normal",
    )

    plt.subplots_adjust(
        left=0.055,
        right=0.995,
        top=0.90,
        bottom=0.035,
        wspace=0.035,
        hspace=0.08,
    )

    output_path = output_dir / f"{moment}_comparison_2x3.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {output_path}")

Saved: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\dp_termpaper\4. Results\plots\simulation\combined_moment_comparisons\avg_experience_over_age_comparison_2x3.png
Saved: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\dp_termpaper\4. Results\plots\simulation\combined_moment_comparisons\avg_hours_over_age_comparison_2x3.png
Saved: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\dp_termpaper\4. Results\plots\simulation\combined_moment_comparisons\avg_wage_over_age_comparison_2x3.png
Saved: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\dp_termpaper\4. Results\plots\simulation\combined_moment_comparisons\avg_wealth_over_age_comparison_2x3.png
Saved: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\dp_termpaper\4. Results\plots\simulation\combined_moment_comparisons\hours_0_over_age_comparison_2x3.png
Saved: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\dp_termpaper\4. Results\plots\simulation\combined_moment_com